<a href="https://colab.research.google.com/github/agcbls49/THESIS/blob/main/Experiment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INTRODUCTION**
This colab file contains the model refinement and experiment comparison of training the best model at 150 epochs. For more information, you can visit this Github [repository](https://github.com/agcbls49/THESIS).

<br>

**The best model was YOLOv26n using the 80% train, 10% validation and 10% testing data split. The trained model will undergo refinement using 150 epochs in this experiment.**

<br>

---

# **SPECIFICATIONS**

The trained model is the YOLOv26n and has a **80% train, 10% valid and 10% test** split and is trained using the following YOLO specifications:
1. 150 Epochs
2. All images resized to 640x640
3. 16 Batches

# **SETUP**

### Check if the correct runtime is set

The runtime should be A100 if the file is run in Google Colab. This is run to ensure that the runtime used is not CPU or a lower powered GPU in Colab.

In [ ]:
!nvidia-smi

### Mounting the Dataset from Google Drive

This step mounts the Google Drive where the dataset is stored and copies the dataset.zip file to Google Colab's storage allowing the dataset to be used for model training.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Copy dataset.zip from Drive to Colab's local storage
!cp "/content/gdrive/MyDrive/2_THESIS_DATASET/dataset2.zip" /content/

### Unzipping the dataset contents
This step simply unzips the content of dataset.zip in Google Colab's storage.

In [ ]:
# Unzip the dataset folder contents
!unzip -q /content/dataset2.zip -d /content/dataset

### Installing Ultralytics for YOLO
This step installs the ultralytics library which is needed for training YOLO models.

In [ ]:
# Install ultralytics for YOLO
!pip install ultralytics -q

### Creating a YAML file
This step creates a YAML file which is needed for training YOLO models. It contains the path of the unzipped dataset, where the train, validation and test images are located, and the names of the classes along with their class numbers.

In [ ]:
import yaml
import os

data = {
    'path': '/content/dataset/dataset2',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 24,
    'names': [
        'backpack',
        'book',
        'cup',
        'glasses',
        'headset',
        'key',
        'phone',
        'remote',
        'wallet',
        'water bottle',
        'body spray',
        'card',
        'charger',
        'comb',
        'flashlight',
        'glasses case',
        'medicine',
        'nail clipper',
        'shoe',
        'watch',
        'body scrub',
        'cane',
        'lighter',
        'towel'
    ]
}

yaml_path = '/content/data.yaml'

# write file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("written")

# verify file exists
print("exists:", os.path.exists(yaml_path))

# verify size
print("size:", os.path.getsize(yaml_path))

# print contents
with open(yaml_path, 'r') as f:
    content = f.read()

print("CONTENT:")
print(content)

# **RUNNING THE MODEL**

Upload the **best.pt** file of the selected model from the evaluations conducted in Experiments 1 and 2. Based on the evaluation results, the selected model is **YOLOv26n** trained using the **80/10/10 dataset split**. The **best.pt** file produced after the 100-epoch training will be used as the starting checkpoint for model refinement. The refinement process will retrain the model for **150 epochs**, initializing it with the weights and biases learned during the previous training stage.


In [ ]:
from google.colab import files
uploaded = files.upload()

Train the best model at 150 epochs

In [ ]:
!yolo detect train data=/content/data.yaml model=/content/best.pt epochs=150 patience=0 name=yolo26n_150_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolo26n_150_run/weights/best.pt split=test name=yolo26n_150_test

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26n summary (fused): 120 layers, 2,379,516 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2405.9±1677.6 MB/s, size: 2184.3 KB)
val: Scanning /content/dataset/dataset2/labels/test... 792 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 792/792 1.2Kit/s 0.6s
val: New cache created: /content/dataset/dataset2/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.8it/s 13.1s
                   all        792       1432      0.916      0.878      0.913      0.766
              backpack         39         39      0.937      0.974      0.983      0.854
                  book         50        128      0.776      0.719      0.809      0.669
                   cup         83        169      0.864       0.84      0.837      0.652
               glasses         53         53   

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolo26n_150_run/weights/best.pt")
model.export(format="tflite")

# Zip the entire run folder (includes best.pt, best.tflite, results.csv, confusion matrices, curves, etc.)
!zip -r yolo26n_150_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolo26n_150_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolo26n_150_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolo26n_150_test_results.txt")



---

